# Download EBP and FRED data

This notebook is self-contained: it includes the code from `load_ebp.py` and `load_fred.py`, so it can be committed to GitHub and run even if those helper files were not uploaded.

By default it saves CSV files under `data/raw/` relative to the directory where the notebook is run. If you open the notebook from the project root, this should match the expected project structure.

In [1]:
from __future__ import annotations

import json
import os
from io import BytesIO
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT.parent / "raw" / "macro_covariates"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DIR}")

Project root: /homes/heo25/Documents/Decision_Focused_Learning_for_Capital_Allocation/data/notebooks
Raw data directory: /homes/heo25/Documents/Decision_Focused_Learning_for_Capital_Allocation/data/raw/macro_covariates


## EBP loader

Downloads and cleans the Federal Reserve Excess Bond Premium dataset, returning a dataframe with `DATE` and `EBP` columns.

In [2]:
EBP_URL = "https://www.federalreserve.gov/econres/notes/feds-notes/ebp_csv.csv"
DEFAULT_EBP_OUTPUT_PATH = RAW_DIR / "ebp.csv"


def _download_ebp_bytes(url: str) -> bytes:
    """Download the EBP CSV bytes with a browser-like user agent."""
    request = Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0 Safari/537.36"
            )
        },
    )

    try:
        with urlopen(request, timeout=30) as response:
            return response.read()
    except HTTPError as exc:
        raise RuntimeError(f"Federal Reserve returned HTTP {exc.code}") from exc
    except URLError as exc:
        raise RuntimeError(f"Could not download EBP dataset: {exc.reason}") from exc


def _read_ebp_csv(url: str) -> pd.DataFrame:
    """Read the EBP CSV while tolerating metadata rows before the header."""
    csv_bytes = _download_ebp_bytes(url)
    csv_buffer = BytesIO(csv_bytes)

    try:
        raw = pd.read_csv(csv_buffer)
    except pd.errors.ParserError:
        csv_buffer.seek(0)
        raw = pd.read_csv(csv_buffer, header=None)

    if "date" in [str(column).strip().lower() for column in raw.columns]:
        return raw

    csv_buffer.seek(0)
    raw_no_header = pd.read_csv(csv_buffer, header=None)
    header_row = raw_no_header.apply(
        lambda row: row.astype(str).str.strip().str.lower().eq("date").any(),
        axis=1,
    )
    if not header_row.any():
        raise ValueError("Could not identify the EBP CSV header row")

    header_index = header_row.idxmax()
    columns = raw_no_header.iloc[header_index].astype(str).str.strip().tolist()
    cleaned = raw_no_header.iloc[header_index + 1 :].copy()
    cleaned.columns = columns
    return cleaned


def _clean_ebp_dataframe(raw: pd.DataFrame, source_label: str) -> pd.DataFrame:
    """Return a cleaned EBP dataframe with DATE and EBP columns."""
    raw = raw.copy()
    raw.columns = [str(column).strip().lower() for column in raw.columns]

    if "date" not in raw.columns:
        raise ValueError(f"{source_label} EBP data does not contain a date column")
    if "ebp" not in raw.columns:
        raise ValueError(f"{source_label} EBP data does not contain an ebp column")

    cleaned = raw[["date", "ebp"]].rename(columns={"date": "DATE", "ebp": "EBP"})
    cleaned["DATE"] = pd.to_datetime(cleaned["DATE"], errors="coerce")
    cleaned["EBP"] = pd.to_numeric(cleaned["EBP"], errors="coerce")
    cleaned = cleaned.dropna(subset=["DATE"]).sort_values("DATE").reset_index(drop=True)

    if cleaned.empty:
        raise ValueError(f"No valid EBP observations were found in {source_label} data")

    return cleaned


def load_ebp(
    output_path: str | Path = DEFAULT_EBP_OUTPUT_PATH,
    force_download: bool = False,
) -> pd.DataFrame:
    """Load, clean, and save the Excess Bond Premium series."""
    output_path = Path(output_path)
    if output_path.exists() and not force_download:
        raw = pd.read_csv(output_path)
        return _clean_ebp_dataframe(raw, "cached")

    raw = _read_ebp_csv(EBP_URL)
    cleaned = _clean_ebp_dataframe(raw, "downloaded")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cleaned.to_csv(output_path, index=False)
    return cleaned

## FRED loader

Downloads a FRED series either through the official API, if `FRED_API_KEY` is set, or through FRED's public CSV endpoint otherwise. Each series is saved as a normalized two-column CSV with `DATE` and the series ID.

In [11]:
import os
from getpass import getpass

FRED_API_KEY ="f541097e91d41885bd60861bbb53a7d6"

In [4]:
FRED_API_URL = "https://api.stlouisfed.org/fred/series/observations"
FRED_CSV_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv"
DEFAULT_FRED_OUTPUT_DIR = RAW_DIR


from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from io import BytesIO
import time

def _download_from_fred_api(series_id: str, api_key: str) -> pd.DataFrame:
    """Download a FRED series through the official API."""
    query = urlencode(
        {
            "series_id": series_id,
            "api_key": api_key,
            "file_type": "json",
        }
    )
    url = f"{FRED_API_URL}?{query}"

    try:
        with urlopen(url, timeout=30) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        raise RuntimeError(f"FRED API returned HTTP {exc.code} for {series_id}") from exc
    except URLError as exc:
        raise RuntimeError(f"Could not download FRED series {series_id}: {exc.reason}") from exc

    if "observations" not in payload:
        message = payload.get("error_message", "missing observations payload")
        raise RuntimeError(f"FRED API response for {series_id} was invalid: {message}")

    observations = payload["observations"]
    return pd.DataFrame(
        {
            "DATE": [observation["date"] for observation in observations],
            series_id: [observation["value"] for observation in observations],
        }
    )


def _download_from_public_csv(series_id: str) -> pd.DataFrame:
    """Download a FRED series from the public graph CSV endpoint."""
    url = f"{FRED_CSV_URL}?{urlencode({'id': series_id})}"

    try:
        df = pd.read_csv(url)
    except HTTPError as exc:
        raise RuntimeError(f"FRED returned HTTP {exc.code} for series {series_id}") from exc
    except URLError as exc:
        raise RuntimeError(f"Could not download FRED series {series_id}: {exc.reason}") from exc

    return df


def _normalize_fred_dataframe(df: pd.DataFrame, series_id: str) -> pd.DataFrame:
    """Return a clean two-column dataframe with DATE and the FRED series ID."""
    rename_map = {
        column: "DATE"
        for column in df.columns
        if str(column).strip().lower() in {"date", "observation_date"}
    }
    df = df.rename(columns=rename_map)

    if "DATE" not in df.columns:
        raise ValueError(f"Downloaded FRED data for {series_id} does not contain a date column")

    value_columns = [column for column in df.columns if column != "DATE"]
    if series_id in value_columns:
        value_column = series_id
    elif len(value_columns) == 1:
        value_column = value_columns[0]
    else:
        raise ValueError(
            f"Downloaded FRED data for {series_id} must contain exactly one value column"
        )

    cleaned = df[["DATE", value_column]].rename(columns={value_column: series_id})
    cleaned["DATE"] = pd.to_datetime(cleaned["DATE"], errors="coerce")
    cleaned[series_id] = pd.to_numeric(cleaned[series_id].replace(".", pd.NA), errors="coerce")
    cleaned = cleaned.dropna(subset=["DATE"]).sort_values("DATE").reset_index(drop=True)

    if cleaned.empty:
        raise ValueError(f"No valid observations found for FRED series {series_id}")

    return cleaned


def download_fred_series(
    series_id: str,
    output_dir: Path = DEFAULT_FRED_OUTPUT_DIR,
    api_key: str | None = None,
    force_download: bool = False,
) -> Path:
    """Download a FRED series as a normalized CSV and return the saved file path."""
    normalized_series_id = series_id.strip().upper()
    if not normalized_series_id:
        raise ValueError("series_id must not be empty")

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{normalized_series_id}.csv"
    if output_path.exists() and not force_download:
        cached = pd.read_csv(output_path)
        _normalize_fred_dataframe(cached, normalized_series_id)
        return output_path

    api_key = api_key or os.getenv("FRED_API_KEY")
    if api_key:
        raw = _download_from_fred_api(normalized_series_id, api_key)
    else:
        raw = _download_from_public_csv(normalized_series_id)

    cleaned = _normalize_fred_dataframe(raw, normalized_series_id)
    cleaned.to_csv(output_path, index=False)
    return output_path



FRED_CSV_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv"


def _download_from_public_csv(series_id: str, retries: int = 5, timeout: int = 60) -> pd.DataFrame:
    """Download a FRED series from the public CSV endpoint with retries."""
    url = f"{FRED_CSV_URL}?{urlencode({'id': series_id})}"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0 Safari/537.36"
        )
    }

    last_error = None

    for attempt in range(1, retries + 1):
        try:
            request = Request(url, headers=headers)

            with urlopen(request, timeout=timeout) as response:
                csv_bytes = response.read()

            return pd.read_csv(BytesIO(csv_bytes))

        except (HTTPError, URLError, TimeoutError) as exc:
            last_error = exc
            print(f"Attempt {attempt}/{retries} failed for {series_id}: {exc}")

            # Backoff: wait a little longer after each failed attempt
            time.sleep(2 * attempt)

    raise RuntimeError(
        f"Could not download FRED series {series_id} after {retries} attempts"
    ) from last_error

## Run the downloads

Edit `FRED_SERIES_IDS` to include whichever macro-financial covariates you need. The default includes your charge-off series so the notebook does something useful immediately.

In [5]:
from pathlib import Path
import yaml

CONFIG_PATH = Path("../raw/fred_series_config.yaml")

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

fred_codes = [item["fred_id"] for item in config["series"]]

In [6]:
# Set this to True if you want to refresh already-downloaded files.
FORCE_DOWNLOAD = False

# Add or remove FRED series IDs here as needed.
# Examples that may be relevant for macro/credit work:
# "UNRATE", "FEDFUNDS", "T10Y2Y", "CPIAUCSL", "INDPRO", "GDP", "BAA10Y"
FRED_SERIES_IDS = [
    'T10Y2Y',
    'VIXCLS',
    'BAA10Y',
    'DRTSCILM'
    ]

# Optional: set directly here, or set the FRED_API_KEY environment variable outside the notebook.
FRED_API_KEY = os.getenv("FRED_API_KEY")

In [7]:
ebp = load_ebp(DEFAULT_EBP_OUTPUT_PATH, force_download=FORCE_DOWNLOAD)
print(f"Saved {len(ebp):,} EBP rows to {DEFAULT_EBP_OUTPUT_PATH}")
ebp.head()

Saved 641 EBP rows to /homes/heo25/Documents/Decision_Focused_Learning_for_Capital_Allocation/data/raw/macro_covariates/ebp.csv


,DATE,EBP
0,1973-01-01,-0.046885
1,1973-02-01,-0.058168
2,1973-03-01,-0.149616
3,1973-04-01,-0.210314
4,1973-05-01,-0.084324


In [12]:
FRED_API_KEY

'f541097e91d41885bd60861bbb53a7d6'

In [13]:
fred_paths = []
failed_series = []

for series_id in FRED_SERIES_IDS:
    try:
        path = download_fred_series(
            series_id,
            output_dir=DEFAULT_FRED_OUTPUT_DIR,
            api_key=FRED_API_KEY,
            force_download=False,
        )
        fred_paths.append(path)
        print(f"Saved {series_id} to {path}")

    except Exception as exc:
        failed_series.append(series_id)
        print(f"Failed to download {series_id}: {exc}")

print("Failed series:", failed_series)

Saved T10Y2Y to /homes/heo25/Documents/Decision_Focused_Learning_for_Capital_Allocation/data/raw/macro_covariates/T10Y2Y.csv
Saved VIXCLS to /homes/heo25/Documents/Decision_Focused_Learning_for_Capital_Allocation/data/raw/macro_covariates/VIXCLS.csv
Saved BAA10Y to /homes/heo25/Documents/Decision_Focused_Learning_for_Capital_Allocation/data/raw/macro_covariates/BAA10Y.csv
Saved DRTSCILM to /homes/heo25/Documents/Decision_Focused_Learning_for_Capital_Allocation/data/raw/macro_covariates/DRTSCILM.csv
Failed series: []


In [ ]:
fred_paths = []
for series_id in FRED_SERIES_IDS:
    path = download_fred_series(
        series_id,
        output_dir=DEFAULT_FRED_OUTPUT_DIR,
        api_key=FRED_API_KEY,
        force_download=FORCE_DOWNLOAD,
    )
    fred_paths.append(path)
    print(f"Saved {series_id} to {path}")

fred_paths

## Optional: inspect saved files

This cell reads back the downloaded CSVs to confirm the files were written correctly.

In [ ]:
saved_files = sorted(RAW_DIR.glob("*.csv"))
print(f"Found {len(saved_files)} CSV files in {RAW_DIR}")
for path in saved_files:
    print(path.name)

# Preview the first saved CSV, if present.
if saved_files:
    display(pd.read_csv(saved_files[0]).head())

## Optional: create a merged raw panel

This produces `data/raw/macro_panel_raw.csv` by outer-joining all downloaded FRED series and EBP on `DATE`. You can skip this if your preprocessing pipeline handles merging elsewhere.

In [ ]:
dataframes = []

if DEFAULT_EBP_OUTPUT_PATH.exists():
    dataframes.append(pd.read_csv(DEFAULT_EBP_OUTPUT_PATH, parse_dates=["DATE"]))

for path in fred_paths:
    dataframes.append(pd.read_csv(path, parse_dates=["DATE"]))

if dataframes:
    panel = dataframes[0]
    for df in dataframes[1:]:
        panel = panel.merge(df, on="DATE", how="outer")
    panel = panel.sort_values("DATE").reset_index(drop=True)

    panel_path = RAW_DIR / "macro_panel_raw.csv"
    panel.to_csv(panel_path, index=False)
    print(f"Saved merged raw panel to {panel_path}")
    display(panel.head())
else:
    print("No dataframes available to merge.")